# Satellite Data Exploration

This notebook:
1. Searches for SAOCOM scenes over the Pampas AOI via ASF (asf-search)
2. Searches for matching Sentinel-2 scenes (sentinelhub)
3. Shows what's available before downloading anything
4. Optionally downloads a single SAOCOM GRD scene
5. Hits the local tile backend to verify synthetic data renders

**Credentials are read from `../../.env` — never hardcode them here.**

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

# Load credentials from repo root .env (gitignored)
env_path = Path("../../.env").resolve()
assert env_path.exists(), f".env not found at {env_path}"
load_dotenv(env_path)

EARTHDATA_USER = os.environ["EARTHDATA_USERNAME"]
EARTHDATA_PASS = os.environ["EARTHDATA_PASSWORD"]
SH_CLIENT_ID   = os.environ["SH_CLIENT_ID"]
SH_SECRET      = os.environ["SH_CLIENT_SECRET"]

print(f"NASA Earthdata user : {EARTHDATA_USER}")
print(f"Copernicus client   : {SH_CLIENT_ID[:8]}...")

---
## 1. SAOCOM scene search via ASF

In [ ]:
import asf_search as asf

# Pampas AOI bounding box (lon_min, lat_min, lon_max, lat_max)
AOI_WKT = "POLYGON((-65 -38, -57 -38, -57 -30, -65 -30, -65 -38))"

results = asf.search(
    platform=asf.PLATFORM.SAOCOM1A,
    processingLevel=asf.PRODUCT_TYPE.GRD_HD,
    intersectsWith=AOI_WKT,
    start="2024-01-01",
    end="2024-12-31",
)

print(f"SAOCOM 1A GRD scenes found: {len(results)}")

In [ ]:
import pandas as pd

rows = []
for r in results:
    p = r.properties
    rows.append({
        "fileID":      p.get("fileID"),
        "startTime":   p.get("startTime"),
        "stopTime":    p.get("stopTime"),
        "polarization":p.get("polarization"),
        "flightDirection": p.get("flightDirection"),
        "sizeMB":      round(p.get("bytes", 0) / 1e6, 1),
        "url":         p.get("url"),
    })

df_saocom = pd.DataFrame(rows)
df_saocom.sort_values("startTime", inplace=True)
df_saocom.head(10)

Also search SAOCOM 1B:

In [ ]:
results_1b = asf.search(
    platform=asf.PLATFORM.SAOCOM1B,
    processingLevel=asf.PRODUCT_TYPE.GRD_HD,
    intersectsWith=AOI_WKT,
    start="2024-01-01",
    end="2024-12-31",
)
print(f"SAOCOM 1B GRD scenes found: {len(results_1b)}")

---
## 2. Sentinel-2 scene search via sentinelhub

In [ ]:
from sentinelhub import (
    SHConfig, DataCollection, SentinelHubCatalog, CRS,
    BBox, bbox_to_dimensions, SentinelHubRequest, MimeType,
)
from datetime import datetime, timedelta

config = SHConfig()
config.sh_client_id     = SH_CLIENT_ID
config.sh_client_secret = SH_SECRET
config.sh_base_url      = "https://sh.dataspace.copernicus.eu"
config.sh_token_url     = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

catalog = SentinelHubCatalog(config=config)

aoi_bbox = BBox(bbox=[-65, -38, -57, -30], crs=CRS.WGS84)

search_iter = catalog.search(
    DataCollection.SENTINEL2_L2A,
    bbox=aoi_bbox,
    time=("2024-01-01", "2024-12-31"),
    filter="eo:cloud_cover < 20",
    fields={"include": ["id", "properties.datetime", "properties.eo:cloud_cover"], "exclude": []},
)

s2_results = list(search_iter)
print(f"Sentinel-2 L2A scenes (cloud < 20 %): {len(s2_results)}")

In [ ]:
s2_rows = []
for item in s2_results:
    props = item.get("properties", {})
    s2_rows.append({
        "id":         item.get("id"),
        "datetime":   props.get("datetime"),
        "cloud_pct":  props.get("eo:cloud_cover"),
    })

df_s2 = pd.DataFrame(s2_rows).sort_values("datetime")
df_s2.head(10)

---
## 3. Find date pairs (SAOCOM + Sentinel-2 within ±3 days)

In [ ]:
from datetime import datetime, timedelta

def parse_dt(s):
    if not s:
        return None
    try:
        return datetime.fromisoformat(s.replace("Z", "+00:00"))
    except Exception:
        return None

saocom_dates = [
    (r.properties.get("startTime"), r)
    for r in list(results) + list(results_1b)
]

pairs = []
for dt_str, saocom_scene in saocom_dates:
    t0 = parse_dt(dt_str)
    if t0 is None:
        continue
    for row in s2_rows:
        t1 = parse_dt(row["datetime"])
        if t1 is None:
            continue
        if abs((t1 - t0).days) <= 3:
            pairs.append({
                "saocom_date": t0.date(),
                "s2_date":     t1.date(),
                "delta_days":  abs((t1 - t0).days),
                "s2_cloud":    row["cloud_pct"],
                "saocom_id":   saocom_scene.properties.get("fileID"),
                "s2_id":       row["id"],
            })

df_pairs = pd.DataFrame(pairs).sort_values(["delta_days", "s2_cloud"])
print(f"{len(df_pairs)} coincident pairs (SAOCOM + S2 within ±3 days)")
df_pairs.head(10)

---
## 4. (Optional) Download the best SAOCOM scene

Pick the first pair and download to `data/raw/`. Only run this cell when you have ~1–2 GB free.

In [ ]:
# Uncomment to download

# best_saocom_id = df_pairs.iloc[0]["saocom_id"]
# target_scene   = next(r for r in list(results) + list(results_1b)
#                       if r.properties.get("fileID") == best_saocom_id)
#
# session = asf.ASFSession().auth_with_creds(EARTHDATA_USER, EARTHDATA_PASS)
# raw_dir = Path("../../data/raw")
# raw_dir.mkdir(parents=True, exist_ok=True)
#
# print(f"Downloading {best_saocom_id} (~{target_scene.properties.get('bytes',0)/1e6:.0f} MB)...")
# asf.FileDownloadType(target_scene).download(path=str(raw_dir), session=session)
# print("Done.")

---
## 5. Verify synthetic backend tiles are rendering

Requires the backend to be running: `cd Arturo/backend && uvicorn app.main:app --reload`

In [ ]:
import requests
from IPython.display import Image, display

BACKEND = "http://localhost:8000"

# Health check
r = requests.get(f"{BACKEND}/health", timeout=5)
print("Health:", r.json())

# List layers
layers = requests.get(f"{BACKEND}/layers", timeout=5).json()
for f in layers["features"]:
    props = f["properties"]
    print(f"  {props['id']:20s} dates={props.get('dates')}")

In [ ]:
# Fetch one tile per product and show it
import io
from PIL import Image as PILImage
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Zoom 5, tile coords roughly over the Pampas AOI (-65,-38,-57,-30)
Z, X, Y = 5, 11, 20

DATES_TO_SHOW = ["20240101", "20240201", "20240315"]

products = [f["properties"]["id"] for f in layers["features"]]

fig, axes = plt.subplots(len(DATES_TO_SHOW), len(products),
                          figsize=(3 * len(products), 3 * len(DATES_TO_SHOW)))
fig.suptitle("Synthetic tile grid — all products × all dates", fontsize=13)

for col, product in enumerate(products):
    for row, date in enumerate(DATES_TO_SHOW):
        ax = axes[row][col]
        url = f"{BACKEND}/tiles/{product}/{Z}/{X}/{Y}.png?date={date}"
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            img = PILImage.open(io.BytesIO(resp.content))
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, f"HTTP {resp.status_code}",
                    ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"{product}\n{date}", fontsize=7)
        ax.axis("off")

plt.tight_layout()
plt.show()

---
## 6. Quick look at a synthetic COG directly

No backend needed — reads the file from disk.

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("../../data/processed").resolve()

products_to_plot = ["soil_moisture", "ndvi", "bsi", "backscatter_hh"]
dates_to_plot    = ["20240101", "20240201", "20240315"]

fig, axes = plt.subplots(len(dates_to_plot), len(products_to_plot),
                          figsize=(4 * len(products_to_plot), 3 * len(dates_to_plot)))
fig.suptitle("Synthetic COG fields — direct disk read", fontsize=13)

for col, product in enumerate(products_to_plot):
    for row, date in enumerate(dates_to_plot):
        ax = axes[row][col]
        pattern = list(DATA_DIR.glob(f"{product}_{date}_*.tif"))
        if not pattern:
            ax.text(0.5, 0.5, "not found", ha="center", va="center",
                    transform=ax.transAxes)
            ax.axis("off")
            continue
        with rasterio.open(pattern[0]) as src:
            data = src.read(1, masked=True)
        im = ax.imshow(data, cmap="viridis", interpolation="nearest")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(f"{product}\n{date}", fontsize=8)
        ax.axis("off")

plt.tight_layout()
plt.show()

---
## Next steps

Once you have a SAOCOM GRD scene in `data/raw/`:

1. Run ESA SNAP preprocessing (calibration → terrain correction → speckle filter)
2. Export backscatter σ⁰ (dB) as a GeoTIFF
3. Download matching Sentinel-2 and compute NDVI, NDMI, BSI
4. Run Water Cloud Model to derive soil moisture
5. Run K-means clustering on fused SAR + optical stack
6. Export each product as a COG to `data/processed/<product>_<date>_<tile>.tif`

The backend will pick up the new files automatically — no restart needed.